# Negation-pair test: a transparent walkthrough

This notebook reproduces Q2-T1 step by step. Its central question is:

> If two quantifiers divide the same scenes in exactly the same way, but swap which side is called true, does monotonicity direction still predict which one is harder to learn?

**The most important provenance fact:** this notebook trains no neural network. It checks symbolic meanings, reuses outcomes from earlier neural training runs, and fits one descriptive regression.

## First, disambiguate the word "model"

| Term in the project | What it is | What this notebook does with it |
|---|---|---|
| **LSTM / Transformer model** | A neural classifier trained to predict a quantifier's truth from a scene | **Not trained here.** We load its previously recorded validation-loss AUC. |
| **`QuantifierModel`** | One set-theoretic scene `<M, A, B>`, encoded as a length-16 sequence | We generate scenes and execute symbolic grammar expressions on them. It is not a learned model. |
| **OLS model** | A linear regression of learning difficulty on upward and downward degree | We fit it here to quantify what the sample-wide directional trend predicts inside the pairs. |

The original learners were a 3-layer LSTM with 20 hidden units and a 2-layer Transformer (`d_model=12`, 4 heads). Both received one-hot scene sequences and were optimized with Adam and `BCEWithLogitsLoss`.

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

# Find src/examples/learn_quant whether Jupyter started there or in notebooks/.
start = Path.cwd().resolve()
ROOT = next(
    path for path in [start, *start.parents]
    if (path / "scripts" / "negation_pair_test.py").exists()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))

import negation_pair_test as t1

CSV = ROOT / "outputs" / "combined_runs_AOC_monotonicity_updated.csv"
ARCHIVE = t1.DEFAULT_ALTK_ARCHIVE
N_VERIFY = 20_000

print("analysis root:", ROOT)
print("archived expression data:", ARCHIVE)
print("archive available:", ARCHIVE.exists())

analysis root: /Users/radmajik/Documents/UWLing/ultk-fresh/src/examples/learn_quant
archived expression data: /Users/radmajik/Documents/UWLing/altk/src/examples
archive available: True


## 1. Inspect the reused neural results

The CSV has one row per completed or attempted training run. `val_loss_step_AOC` is accumulated validation loss over training: larger values mean the learner remained wrong for longer. We retain completed runs, then average repeated runs within expression and architecture.

In [2]:
raw_runs = pd.read_csv(CSV)
runs = t1.load_runs(CSV)

display(pd.DataFrame({
    "quantity": [
        "all CSV rows",
        "completed rows used here",
        "unique expressions with a completed run",
        "neural architectures",
    ],
    "value": [
        len(raw_runs),
        len(runs),
        runs["expression"].nunique(),
        ", ".join(sorted(runs["model"].unique())),
    ],
}))

display(
    runs.groupby("model").agg(
        completed_rows=("val_loss_step_AOC", "size"),
        expressions=("expression", "nunique"),
        mean_auc=("val_loss_step_AOC", "mean"),
    ).round(1)
)

,quantity,value
0,all CSV rows,8005
1,completed rows used here,7172
2,unique expressions with a completed run,1795
3,neural architectures,"LSTM, Transformer"


,completed_rows,expressions,mean_auc
model,,,
LSTM,3587,1792,2358.4
Transformer,3585,1795,3325.1


## 2. Find candidate complement pairs on the finite semantic universe

The archive contains 9,550 unique grammar-generated meanings. Each meaning is represented by 256 truth values, one for each scene in the `M4/X4` universe used to calculate monotonicity. Two meanings are candidates when every bit in one vector is the opposite of the corresponding bit in the other vector.

We then restrict the search to the 2,000 expressions that actually entered neural training.

In [3]:
if not ARCHIVE.exists():
    raise FileNotFoundError(
        f"Original archive not found at {ARCHIVE}. Set ARCHIVE to the old "
        "altk/src/examples directory before continuing."
    )

terms, truth_matrix, universe, expressions = t1.load_archive(ARCHIVE)
trained_sample = pd.read_csv(ROOT / "expressions_sample_2k.csv")
candidate_pairs, trained_indices = t1.find_pairs(
    terms, truth_matrix, trained_sample["term_expression"]
)

print("truth matrix shape:", truth_matrix.shape)
print("candidate complement pairs:", len(candidate_pairs))

archive pool: 9550 unique meanings on 256 models
pool meanings with complement in pool:        884
trained expressions with complement in pool:  269 / 2000
complement pairs within the trained 2000:     44
truth matrix shape: (9550, 256)
candidate complement pairs: 44


### Look at one candidate directly

The next table exposes the test rather than hiding it behind a count. For each of the first 16 finite-universe scenes, `q1` and `q2` must disagree. `opposites` should therefore be `True` in every row.

In [4]:
i, j = candidate_pairs[0]
print("expression 1:", terms[i])
print("expression 2:", terms[j])

display(pd.DataFrame({
    "finite_universe_scene": [ref.name for ref in universe.referents[:16]],
    "q1": truth_matrix[i, :16],
    "q2": truth_matrix[j, :16],
    "opposites": truth_matrix[i, :16] == ~truth_matrix[j, :16],
}))

expression 1: subset_eq(index(cardinality(A), difference(B, A)), A)
expression 2: greater_than(cardinality(difference(B, A)), cardinality(A))


,finite_universe_scene,q1,q2,opposites
0,0102,True,False,True
1,0210,True,False,True
2,0301,True,False,True
3,2221,True,False,True
4,0112,True,False,True
5,1100,True,False,True
6,0020,True,False,True
7,3122,True,False,True
8,2213,True,False,True
9,2320,True,False,True


## 3. Verify candidates on training-style scenes

A pair can be complementary on the 256-scene measurement universe but disagree elsewhere. The actual neural training distribution used length-16 scenes with `|M|=12` and `inclusive=False`. We therefore generate 20,000 scenes with those settings and execute both symbolic expressions on every scene.

This is the only sense in which a `QuantifierModel` is "run": a grammar expression is called as a deterministic function of a scene. No weights are learned.

In [5]:
verified_pairs, failed_pairs = t1.verify_pairs(
    candidate_pairs, expressions, ARCHIVE, N_VERIFY, seed=7
)

display(pd.DataFrame({
    "stage": ["candidate pairs on 256 scenes", "verified on 20,000 scenes", "failed verification"],
    "pairs": [len(candidate_pairs), len(verified_pairs), len(failed_pairs)],
}))

functional verification on 20000 training-style scenes (M=12, X=16): 41 verified, 3 failed
  FAILED (2757 disagreements): and(not(greater_than(cardinality(B), cardinality(A))), greater_than(ca  vs  or(equals(cardinality(intersection(A, B)), cardinality(difference(A, B
  FAILED (4422 disagreements): or(and(subset_eq(B, A), not(subset_eq(A, B))), greater_than(cardinalit  vs  or(and(not(subset_eq(B, A)), greater_than(cardinality(A), cardinality(
  FAILED (4463 disagreements): or(and(subset_eq(A, B), not(subset_eq(B, A))), greater_than(cardinalit  vs  or(and(not(subset_eq(A, B)), greater_than(cardinality(B), cardinality(


,stage,pairs
0,candidate pairs on 256 scenes,44
1,"verified on 20,000 scenes",41
2,failed verification,3


## 4. Join each verified pair to its learning outcomes

For each expression we calculate:

- mean LSTM AUC across completed repeated runs;
- mean Transformer AUC across completed repeated runs;
- `upward = max(left_upward, right_upward)`;
- `downward = max(left_downward, right_downward)`.

Within each complement pair, **D** is whichever member has the larger `downward - upward` score; **U** is the other member. This naming is relative: some weak-contrast pairs are not clean textbook downward/upward pairs.

In [6]:
expression_stats = t1.per_expression_auc(runs)
pairs = t1.build_pair_table(verified_pairs, terms, expression_stats)
pairs["delta_auc_lstm_U_minus_D"] = pairs["auc_lstm_U"] - pairs["auc_lstm_D"]
pairs["delta_auc_tf_U_minus_D"] = pairs["auc_tf_U"] - pairs["auc_tf_D"]
pairs["delta_auc_mean_U_minus_D"] = pairs["auc_mean_U"] - pairs["auc_mean_D"]

print("verified semantic pairs:", len(verified_pairs))
print("pairs with reusable neural outcomes for both members:", len(pairs))

display(
    pairs.sort_values("polarity_contrast", ascending=False)[[
        "term_D", "term_U", "down_D", "up_D", "down_U", "up_U",
        "polarity_contrast", "auc_lstm_D", "auc_lstm_U",
        "auc_tf_D", "auc_tf_U", "delta_auc_mean_U_minus_D",
    ]].head(12).round(2)
)

verified semantic pairs: 41
pairs with reusable neural outcomes for both members: 36


,term_D,term_U,down_D,up_D,down_U,up_U,polarity_contrast,auc_lstm_D,auc_lstm_U,auc_tf_D,auc_tf_U,delta_auc_mean_U_minus_D
5,"subset_eq(A, difference(A, B))","not(subset_eq(A, difference(A, B)))",1.00,0.00,0.12,1.00,1.88,211.85,214.70,160.92,146.63,-5.72
34,"or(subset_eq(B, difference(A, A)), greater_tha...","and(not(subset_eq(B, difference(A, A))), subse...",1.00,0.00,0.62,1.00,1.38,447.34,478.78,801.49,746.02,-12.02
28,"and(and(not(subset_eq(B, A)), subset_eq(A, dif...","subset_eq(index(cardinality(intersection(A, B)...",0.96,0.38,0.28,0.94,1.24,843.54,846.28,1062.88,741.84,-159.15
27,"subset_eq(index(cardinality(B), A), index(card...","and(greater_than(cardinality(A), cardinality(B...",1.00,0.00,0.79,1.00,1.21,402.79,475.99,454.81,548.28,83.34
33,"subset_eq(intersection(A, B), difference(A, in...","and(not(subset_eq(B, A)), subset_eq(index(card...",0.49,0.00,0.46,0.77,0.79,3074.23,3563.32,5107.02,5516.69,449.38
21,"not(subset_eq(index(cardinality(B), union(A, B...","subset_eq(index(cardinality(B), union(A, B)), ...",0.70,0.44,0.33,0.65,0.58,3172.43,3011.29,5477.11,5668.23,14.99
6,"subset_eq(difference(B, A), index(cardinality(...","not(subset_eq(difference(B, A), index(cardinal...",1.00,0.61,0.88,1.00,0.51,245.86,236.48,167.95,206.47,14.57
26,"subset_eq(difference(A, B), index(cardinality(...","not(subset_eq(difference(A, B), index(cardinal...",1.00,0.61,0.88,1.00,0.51,198.33,213.85,188.66,189.10,7.98
15,"or(or(subset_eq(A, B), subset_eq(B, A)), subse...","not(or(or(subset_eq(A, B), subset_eq(B, A)), s...",0.00,0.00,0.37,0.66,0.29,1059.96,1225.78,1657.00,1421.92,-34.63
24,"subset_eq(index(cardinality(B), union(A, B)), ...","not(subset_eq(index(cardinality(B), union(A, B...",1.00,0.92,0.80,1.00,0.28,1539.28,1386.90,1576.91,2106.51,188.61


## 5. Compare difficulty inside the pairs

The estimand is `Delta AUC = AUC(U) - AUC(D)`.

- Positive: the upward/other member was harder.
- Negative: the downward-dominant member was harder.
- Near zero: swapping the labels did not materially change difficulty.

The Wilcoxon signed-rank test asks whether the paired differences are systematically displaced from zero.

In [7]:
def paired_summary(label, col_d, col_u, frame=pairs):
    data = frame[[col_d, col_u]].dropna()
    delta = data[col_u] - data[col_d]
    return {
        "architecture": label,
        "pairs": len(data),
        "mean_AUC_D": data[col_d].mean(),
        "mean_AUC_U": data[col_u].mean(),
        "mean_delta_U_minus_D": delta.mean(),
        "median_delta": delta.median(),
        "fraction_U_harder": (delta > 0).mean(),
        "wilcoxon_p": stats.wilcoxon(delta).pvalue,
    }

paired_results = pd.DataFrame([
    paired_summary("LSTM", "auc_lstm_D", "auc_lstm_U"),
    paired_summary("Transformer", "auc_tf_D", "auc_tf_U"),
    paired_summary("mean of architectures", "auc_mean_D", "auc_mean_U"),
])
display(paired_results.round(3))

,architecture,pairs,mean_AUC_D,mean_AUC_U,mean_delta_U_minus_D,median_delta,fraction_U_harder,wilcoxon_p
0,LSTM,36,1024.072,1089.663,65.592,21.448,0.639,0.162
1,Transformer,36,1622.986,1589.254,-33.731,-17.666,0.417,0.405
2,mean of architectures,36,1323.529,1339.459,15.930,-1.249,0.500,0.592


## 6. Where does the 0.12 ratio come from?

This section constructs a **benchmark**, not another neural experiment. Before doing any arithmetic, define the terms:

- **Downward degree** and **upward degree** are scores from 0 to 1. A score of 1 means perfectly monotone in that direction; 0 means no measured monotonicity in that direction. Each is the larger of the expression's left-argument and right-argument scores.
- **Directional leaning** is `downward degree - upward degree`. It ranges from +1 (fully downward, not upward) to -1 (fully upward, not downward).
- **D member** is the complement-pair member with the larger directional leaning. **U member** is the other. These are relative labels; weak pairs need not contain one textbook-downward and one textbook-upward expression.
- **Polarity contrast** is `leaning(D) - leaning(U)`. It says how differently the monotonicity measure describes the two members.
- **Strong-contrast pair** means `polarity contrast > 0.20`. The 0.20 cutoff is an operational sensitivity cutoff, not a theorem or a learned threshold. It excludes pairs whose directional scores barely differ, because they cannot strongly test a direction-based prediction. Eleven of the 36 analyzable pairs meet it.
- **Standard deviation (SD)** describes the observed spread of a score across all expressions. Converting a difference to SD units lets downward and upward scores enter the regression on comparable scales.
- **Regression coefficient** is the estimated AUC change for a one-SD score increase while holding the other score fixed.
- **Predicted gap** is the regression's expected `AUC(U) - AUC(D)` for a pair. **Observed gap** is the actual archived `AUC(U) - AUC(D)`.

### Step 1: estimate the sample-wide rule of thumb

Across all 1,795 expressions with outcomes, fit `mean AUC ~ standardized downward + standardized upward`. "Standardized" means subtracting the sample mean and dividing by the score's SD. The intercept is not relevant to a within-pair difference because it cancels when U is subtracted from D.

The fitted coefficients are approximately **-708 AUC per SD of downward degree** and **+20 AUC per SD of upward degree**. In plain language: across the full sample, a typical-sized increase in downward degree is associated with 708 less accumulated loss (easier learning); upward degree contributes almost nothing.

### Step 2: use that rule to predict each pair's gap

For each pair, calculate `U - D` separately for downward and upward degree. Divide each difference by that score's SD, multiply by its regression coefficient, and add the two contributions:

`predicted gap = -708 x (down_U - down_D) / SD(down) + 20 x (up_U - up_D) / SD(up)`

This is the AUC gap expected **if the full-sample direction association also operates when the semantic boundary is held fixed**.

### Step 3: compare the prediction with the archived outcomes

For each pair, the observed gap is simply `mean AUC(U) - mean AUC(D)`. We average predicted gaps across the 11 strong-contrast pairs, separately average observed gaps across those same pairs, then divide the averages. A ratio near 1 would mean the predicted direction gap appeared inside the controlled pairs. A ratio near 0 means it largely vanished.

**Important:** 0.12 is not accuracy, variance explained, or a p-value. It is `mean observed difficulty gap / mean predicted difficulty gap`.

In [8]:
population = expression_stats.dropna(subset=["auc_mean", "downward", "upward"])
sd_down = population["downward"].std(ddof=0)
sd_up = population["upward"].std(ddof=0)

X = sm.add_constant(pd.DataFrame({
    "downward_z": (population["downward"] - population["downward"].mean()) / sd_down,
    "upward_z": (population["upward"] - population["upward"].mean()) / sd_up,
}))
ols = sm.OLS(population["auc_mean"], X).fit()

display(pd.DataFrame({
    "term": ["downward degree", "upward degree"],
    "score SD": [sd_down, sd_up],
    "AUC change per +1 SD": [ols.params["downward_z"], ols.params["upward_z"]],
    "plain reading": [
        "More downward predicts less accumulated loss (easier)",
        "Upward degree has almost no sample-wide association",
    ],
}).round(3))

pairs["downward_difference_U_minus_D"] = pairs["down_U"] - pairs["down_D"]
pairs["upward_difference_U_minus_D"] = pairs["up_U"] - pairs["up_D"]
pairs["downward_difference_in_SD"] = pairs["downward_difference_U_minus_D"] / sd_down
pairs["upward_difference_in_SD"] = pairs["upward_difference_U_minus_D"] / sd_up
pairs["predicted_from_downward"] = ols.params["downward_z"] * pairs["downward_difference_in_SD"]
pairs["predicted_from_upward"] = ols.params["upward_z"] * pairs["upward_difference_in_SD"]
pairs["predicted_delta"] = pairs["predicted_from_downward"] + pairs["predicted_from_upward"]
pairs["observed_delta"] = pairs["auc_mean_U"] - pairs["auc_mean_D"]

# Work through the highest-contrast pair: "No A are B" vs "Some A are B".
example = pairs.sort_values("polarity_contrast", ascending=False).iloc[0]
display(pd.DataFrame({
    "quantity": [
        "D expression", "U expression", "D directional leaning",
        "U directional leaning", "polarity contrast",
        "downward U-D", "downward U-D in SD", "predicted downward contribution",
        "upward U-D", "upward U-D in SD", "predicted upward contribution",
        "total predicted AUC gap (U-D)", "D mean AUC", "U mean AUC",
        "observed AUC gap (U-D)",
    ],
    "value": [
        example["term_D"], example["term_U"],
        example["down_D"] - example["up_D"],
        example["down_U"] - example["up_U"], example["polarity_contrast"],
        example["downward_difference_U_minus_D"], example["downward_difference_in_SD"],
        example["predicted_from_downward"], example["upward_difference_U_minus_D"],
        example["upward_difference_in_SD"], example["predicted_from_upward"],
        example["predicted_delta"], example["auc_mean_D"], example["auc_mean_U"],
        example["observed_delta"],
    ],
}))

contrast_pairs = pairs[pairs["polarity_contrast"] > 0.2].dropna(
    subset=["predicted_delta", "observed_delta"]
)
predicted = contrast_pairs["predicted_delta"].mean()
observed = contrast_pairs["observed_delta"].mean()
ratio = observed / predicted

display(pd.DataFrame({
    "quantity": [
        "strong-contrast pairs", "mean predicted AUC gap (U-D)",
        "mean observed AUC gap (U-D)", "observed / predicted",
    ],
    "value": [len(contrast_pairs), predicted, observed, ratio],
    "meaning": [
        "Pairs whose polarity contrast is greater than 0.20",
        "Expected gap if the sample-wide direction rule transfers into pairs",
        "Gap in the archived neural outcomes for those same pairs",
        "65.5 / 558.4 = 0.117, rounded to 0.12",
    ],
}).round(3))

,term,score SD,AUC change per +1 SD,plain reading
0,downward degree,0.303,-707.997,More downward predicts less accumulated loss (...
1,upward degree,0.341,20.104,Upward degree has almost no sample-wide associ...


,quantity,value
0,D expression,"subset_eq(A, difference(A, B))"
1,U expression,"not(subset_eq(A, difference(A, B)))"
2,D directional leaning,1.0
3,U directional leaning,-0.877349
4,polarity contrast,1.877349
5,downward U-D,-0.877349
6,downward U-D in SD,-2.892413
7,predicted downward contribution,2047.820895
8,upward U-D,1.0
9,upward U-D in SD,2.929976


,quantity,value,meaning
0,strong-contrast pairs,11.000,Pairs whose polarity contrast is greater than ...
1,mean predicted AUC gap (U-D),558.446,Expected gap if the sample-wide direction rule...
2,mean observed AUC gap (U-D),65.457,Gap in the archived neural outcomes for those ...
3,observed / predicted,0.117,"65.5 / 558.4 = 0.117, rounded to 0.12"


### Read the worked pair in ordinary language

The displayed example is **"No A are B"** (`subset_eq(A, difference(A, B))`) versus its complement **"Some A are B"**. The measure calls "No" strongly downward (down = 1.000, up = 0.000) and "Some" strongly upward (down = 0.123, up = 1.000), so their polarity contrast is 1.877 and they comfortably pass the 0.20 strong-contrast cutoff.

The full-sample regression converts their score differences into a prediction that U should accumulate about **2,107 more AUC points** than D—that is, "Some" should be much harder. But their archived mean AUCs are 180.7 for U and 186.4 for D, an observed U-D gap of **-5.7**. They were effectively equally easy, with U marginally easier rather than dramatically harder.

The headline ratio does **not** divide -5.7 by 2,107 for this one pair. The same calculation is performed for all 11 strong-contrast pairs. Their predicted gaps average **+558.4**; their observed gaps average **+65.5**. Therefore `65.5 / 558.4 = 0.117`, reported as **0.12**.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for col_d, col_u, color, label in [
    ("auc_lstm_D", "auc_lstm_U", "#e8a33d", "LSTM"),
    ("auc_tf_D", "auc_tf_U", "#4477aa", "Transformer"),
]:
    axes[0].scatter(pairs[col_d], pairs[col_u], alpha=0.75, color=color, label=label)
limit = np.nanmax(pairs[["auc_lstm_D", "auc_lstm_U", "auc_tf_D", "auc_tf_U"]].to_numpy()) * 1.05
axes[0].plot([0, limit], [0, limit], "k--", linewidth=1)
axes[0].set(xlabel="AUC of D member", ylabel="AUC of U member", title="Same boundary: are both sides equally hard?")
axes[0].legend()

axes[1].scatter(pairs["predicted_delta"], pairs["observed_delta"], color="#4f3d8b", alpha=0.8)
bound = max(pairs["predicted_delta"].abs().max(), pairs["observed_delta"].abs().max()) * 1.05
axes[1].plot([-bound, bound], [-bound, bound], "k--", linewidth=1)
axes[1].axhline(0, color="0.7", linewidth=1)
axes[1].set(xlabel="Gap predicted from population regression", ylabel="Gap observed within pair", title="Population trend does not transfer into pairs")

fig.tight_layout()
plt.show()

/var/folders/ww/zmy65s114cv2_jtlds20mdzc0000gn/T/ipykernel_66479/2061199539.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Sanity-check the monotonicity measure itself

For exact complements, upward degree for one member should mirror downward degree for the other. Large failures of `up(Q) = down(not Q)` indicate that the finite-universe directional score is not perfectly respecting the theoretical complement identity.

In [10]:
mirror_errors = pd.concat([
    pairs["mirror_up_D_vs_down_U"].abs(),
    pairs["mirror_down_D_vs_up_U"].abs(),
], ignore_index=True)

display(pd.DataFrame({
    "mean absolute mismatch": [mirror_errors.mean()],
    "median absolute mismatch": [mirror_errors.median()],
    "maximum mismatch": [mirror_errors.max()],
    "fraction above 0.05": [(mirror_errors > 0.05).mean()],
}).round(4))

,mean absolute mismatch,median absolute mismatch,maximum mismatch,fraction above 0.05
0,0.1815,0.0482,0.7885,0.5


## Interpretation

The neural learners had already been trained independently on both members of each pair. This notebook does not recreate that expensive stage; it creates a controlled comparison from those outcomes.

The expected reproduced result is:

- 44 candidate pairs on the 256-scene universe;
- 41 remain exact complements on 20,000 training-style scenes;
- 36 have complete neural outcomes for both members;
- their mean AUC difference is near zero and not statistically distinguishable from zero;
- among 11 strong-direction-contrast pairs, the observed gap is about 0.12 of the population-regression prediction.

**Conclusion:** the large sample-wide association between downward monotonicity and easier learning is not reproduced when the semantic boundary is held fixed and only the truth labels are swapped. The 0.12 ratio does not mean that a new model achieved 12% accuracy. It means that the paired difficulty gap was only 12% as large as a sample-wide regression would have predicted.

**Limits:** this is an observational paired analysis of previously trained expressions, only 11 pairs have a strong directional contrast, pair members need not have equal formula length, and complement status is empirically checked on 20,000 sampled training-style scenes rather than proven over the entire larger scene space.